# Project 1 Overview 


Source of data: Ebay website 

Type of product: laptop

Information extracted: Title, Price, Link to product page, Subtitle

Data collection Info: Data was extracted on Oct 22 2025. Dataset has 300 Observations(product info from the first 5 pages)

# Import Packages

In [1]:
#------------------- IMPORT PACKAGES FOR DATA PROCESSING ----------------------#

# Manage datasets
import pandas as pd

# Work with time data
import time 

# Conduct HTTP requests
import requests

# Construct tree structure of HTML data
import html5lib

# Parse HTML data obtained from scraping
from bs4 import BeautifulSoup

# Import webdriver for chrome
from webdriver_manager.chrome import ChromeDriverManager


from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager


# Automate navigating within browser (SELENIUM)
#------ Key: Manage keys
#------ Select: Obtain features from website
#------ WebDriverWait: Add wait times implicitly
#------ By: Use common information locator strategies
#------ EC and Options: Browser configuration
#------ remote.command: Check whether browser is active

from selenium import webdriver #to automate the navigating within the browser
from selenium.webdriver.chrome.service import service
from selenium.webdriver.common.keys    import Keys
from selenium.webdriver.support.ui     import Select
from selenium.webdriver.support.ui     import WebDriverWait 
from selenium.webdriver.common.by      import By
from selenium.webdriver.support        import expected_conditions as EC
from selenium.webdriver.chrome.options import Options #to use properties of the chrome webbrowser
from selenium.webdriver.remote.command import Command # Use to check whether the web driver is active

In [2]:

opts = Options()                 
driver = webdriver.Chrome(options=opts)


# Data source is search results of "laptop" on ebay

In [3]:
query = "laptop" 
url = f"https://www.ebay.com/sch/i.html?_nkw={query}" # define the url
driver.get(url)
print(driver.current_url)
print(driver.title)
open("debug_ebay.html","w",encoding="utf-8").write(driver.page_source)  # open the ebay page searching for laptops


https://www.ebay.com/sch/i.html?_nkw=laptop
Laptop for sale | eBay


1992181

In [4]:

wait = WebDriverWait(driver, 10)

    # Try to click down cookie consent pop-up if there is one
try:
        wait.until(EC.element_to_be_clickable((By.ID, "gdpr-banner-accept"))).click()
        time.sleep(0.5)
except Exception:
    pass
try:
        btns = driver.find_elements(By.XPATH, '//button[contains(., "Accept")]')
        if btns: btns[0].click()
except Exception:
    pass

In [ ]:
dataset = []   # create a dataset to store collected information
for page in range(0,5): # extract information from first 5 pages

    # wait until result cards are shown
    wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "#srp-river-results li.s-item, #srp-river-results li.s-card")
    ))
    items = driver.find_elements(By.CSS_SELECTOR, "#srp-river-results li.s-item, #srp-river-results li.s-card") #get all the cards
    print(f"Page {page+1}: found {len(items)} items")
    # extract information (title, price, link and subtitle) from each card on the first page
    for it in items:
        # title, price
        title_el = it.find_elements(By.CSS_SELECTOR, ".s-item__title, .s-card__title")
        price_el = it.find_elements(By.CSS_SELECTOR, ".s-item__price, .s-card__price")
        
        title = title_el[0].text.strip() if title_el else None
        price = price_el[0].text.strip() if price_el else None
        

        if not title or title.lower().startswith("shop on ebay"): # skip cards with no titile
            continue
        # link
        link = title_el[0].find_element(By.XPATH, "./ancestor::a[1]").get_attribute("href")
        # subtitle
        subtitle_el = it.find_elements(By.CSS_SELECTOR,  ".s-card__subtitle-row .su-styled-text.secondary.default")
        subtitle = " ".join([s.text.strip() for s in subtitle_el if s.text.strip()])
        # save to dataset
        dataset.append({"title": title, "price": price, "subtitle": subtitle,"link": link,})
    # find the next page button
    sent = items[0] if items else None  # remember one element on the page
    next = driver.find_elements(By.CSS_SELECTOR, "nav.pagination a[type='next']") 
    if not next: break                                               # ends if no next page
    next[0].click()                                                  # click next page button
    if sent: WebDriverWait(driver, 20).until(EC.staleness_of(sent)) # wait page to refresh

pd.DataFrame(dataset).to_csv("ebay_results.csv", index=False, encoding="utf-8-sig")
print("Saved:", len(dataset))
    

Page 1: found 60 items
Page 2: found 60 items
Page 3: found 60 items
Page 4: found 60 items
Page 5: found 60 items
Saved: 300


I collected information(title, price, link and subtitle) from the first 5 pages on ebay and save them to a dataset

In [22]:
# Time
from datetime import datetime
scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("Scraped at:", scraped_at)

Scraped at: 2025-10-22 18:35:25


In [20]:
df= pd.DataFrame(dataset)
df.head(5)

,title,price,subtitle,link
0,Panasonic CF-20 Core M5-6Y57 8GB RAM 256GB Har...,$122.22,GOOD CONDITION Pre-Owned · Panasonic · 256 GB,https://www.ebay.com/itm/286823939471?_skw=lap...
1,"Dell Latitude Laptop PC 11.6"" HD Intel Celeron...",$91.17,Good - Refurbished · Dell,https://www.ebay.com/itm/254170214367?_skw=lap...
2,"HP EliteBook 840 G7 14"" Touchscreen Laptop, In...",$333.00,Pre-Owned · HP,https://www.ebay.com/itm/197720826902?_skw=lap...
3,Dell Latitude 3190 Laptop Computer PC Intel Ce...,$108.44,Good - Refurbished · Dell,https://www.ebay.com/itm/255080719892?_skw=lap...
4,Dell Latitude Laptop Computer PC Intel i5 Up T...,$237.36,Genuine Windows 11 OS! Great Condition! Fast S...,https://www.ebay.com/itm/286393092388?_skw=lap...


# Check dataset and have a short analysis 

In [25]:
df.empty

False

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     300 non-null    object
 1   price     300 non-null    object
 2   subtitle  300 non-null    object
 3   link      300 non-null    object
dtypes: object(4)
memory usage: 9.5+ KB


In [26]:
# convert price to numeric type
df['price'] = df['price'].str.replace('$', '', regex=False)  # delete the $ before number
df['price'] = pd.to_numeric(df['price'], errors='coerce')  
print(df['price'].dtype)

float64


In [ ]:
print(df['price'].agg(['mean', 'var', 'std']))


mean      247.926148
var     22084.078460
std       148.607128
Name: price, dtype: float64

The price mean: 247.9

The price variance: 22084.1

The price standard deviation:148.6


# extract Info from each card's page

In [5]:
wait = WebDriverWait(driver, 10)

wait.until(EC.visibility_of_element_located(
    (By.CSS_SELECTOR, "#srp-river-results li.s-item, #srp-river-results li.s-card")
))
# wait until cards are shown
cards = driver.find_elements(By.CSS_SELECTOR, 
                             "#srp-river-results li.s-item, #srp-river-results li.s-card")[:10] 
# get the first 10 cards


In [13]:
# define function that extract the value on the right side of "label"

def get_spec(label):
    try:
        xp = f'//dt[.//span[normalize-space()="{label}"]]/following-sibling::dd[1]'
        el = WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.XPATH, xp)))
        return el.text.strip()
    except:
        return None
    
rows = [] 

for idx, it in enumerate(cards, 1):
    # 2) get title and link of each card
    title_el = it.find_elements(By.CSS_SELECTOR, ".s-item__title, .s-card__title")
    if not title_el: 
        continue
    title = title_el[0].text.strip() # get title
    if not title or title.lower().startswith("shop on ebay"):
        continue
    link = title_el[0].find_element(By.XPATH, "./ancestor::a[1]").get_attribute("href") # get link
    if not link:
        continue

    # 3) 在新标签打开详情页，抓字段，然后关掉返回
    driver.execute_script("window.open(arguments[0], '_blank');", link)
    driver.switch_to.window(driver.window_handles[-1])  # 切到新标签
    try:
        condition = get_spec("Condition")
        processor = get_spec("Processor")
        ssd = get_spec("SSD Capacity")
        gpu = get_spec("GPU")
        releaseyear = get_spec("Release Year")
        size = get_spec("Screen Size")
        ram = get_spec("RAM Size")
        model = get_spec("Model")
        
        # 如需更多字段，照着写一行即可： e.g. os_ = get_spec("Operating System")

        rows.append({"title": title, "link": link, "condition": condition, "processor": processor,
                     "ssd":ssd, "gpu":gpu, "releaseyear":releaseyear, "size":size, "ram":ram, "model":model})
    finally:
        driver.close()                                  # 关掉详情页
        driver.switch_to.window(driver.window_handles[0])  # 回到搜索结果页

# 4) rows 里就是前 10 个详情的关键信息

pd.DataFrame(rows).to_csv("ebay_detail_sample.csv", index=False, encoding="utf-8-sig")
print("saved:", len(rows))

saved: 10


# Combined Codes: first 2 pages

In [5]:
wait = WebDriverWait(driver, 10)

# 读取详情页“Item specifics”的某一项（左 dt 文本 = label，右 dd 为值）
def get_spec(label):
    try:
        xp = f'//dt[.//span[normalize-space()="{label}"]]/following-sibling::dd[1]'
        el = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, xp)))
        return el.text.strip()
    except:
        return None

rows = []
pages_to_get = 2        # 2 pages

for page in range(pages_to_get):
    # 1) 等待本页卡片加载，并取前 60 个
    wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "#srp-river-results li.s-item, #srp-river-results li.s-card")
    ))
    cards = driver.find_elements(By.CSS_SELECTOR,
        "#srp-river-results li.s-item, #srp-river-results li.s-card") # get all cards on the page
    print(f"Page {page+1}: {len(cards)} cards")

    # 2) 逐个进入详情页抓字段
    for it in cards:
        # link and price
        title_el = it.find_elements(By.CSS_SELECTOR, ".s-item__title, .s-card__title")
        price_el = it.find_elements(By.CSS_SELECTOR, ".s-item__price, .s-card__price")
        
        title = title_el[0].text.strip() if title_el else None
        price = price_el[0].text.strip() if price_el else None
        
        if not title or title.lower().startswith("shop on ebay"):
            continue
        # link
        try:
            link = title_el[0].find_element(By.XPATH, "./ancestor::a[1]").get_attribute("href")
        except:
            link = None
        if not link:
            continue

        # 新标签打开详情 → 抓取 → 关闭返回
        driver.execute_script("window.open(arguments[0], '_blank');", link)
        driver.switch_to.window(driver.window_handles[-1]) # to new tab
        try:
            condition = get_spec("Condition")
            processor = get_spec("Processor")
            ssd       = get_spec("SSD Capacity")
            gpu       = get_spec("GPU")
            releaseyr = get_spec("Release Year")
            size      = get_spec("Screen Size")
            ram       = get_spec("RAM Size")
            model     = get_spec("Model")
            rows.append({
                "title": title, "link": link,
                "condition": condition, "processor": processor,
                "ssd": ssd, "gpu": gpu, "release_year": releaseyr,
                "screen_size": size, "ram": ram, "model": model
            })
        finally:
            driver.close()
            driver.switch_to.window(driver.window_handles[0])

    # 3) 翻到下一页（没有就提前结束）
    nxt = driver.find_elements(By.CSS_SELECTOR, "nav.pagination a[type='next']")
    if page < pages_to_get-1:                # 还需要翻页时才点击
        if not nxt: break
        sentinel = cards[0] if cards else None
        nxt[0].click()
        if sentinel:
            WebDriverWait(driver, 20).until(EC.staleness_of(sentinel))

# 4) 保存
pd.DataFrame(rows).to_csv("ebay_detail_2pages.csv", index=False, encoding="utf-8-sig")
print("saved:", len(rows))


Page 1: 60 cards
Page 2: 60 cards
saved: 120


In [6]:
# Time
from datetime import datetime
scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("Scraped at:", scraped_at)

Scraped at: 2025-10-23 17:25:59
